In [36]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ"
    r"\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist"
    r"\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"
)

In [37]:
from google.cloud import bigquery
from google.oauth2 import service_account

# Google Cloud credentials
credentials = service_account.Credentials.from_service_account_file(

    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"

)

In [38]:
# Create a "Client" object
client = bigquery.Client()

# Construct a reference to the "github_repos" dataset
dataset_ref = client.dataset("github_repos", project="bigquery-public-data")

# API request - fetch the dataset
dataset = client.get_dataset(dataset_ref)

# Construct a reference to the "sample_commits" table
table_ref = dataset_ref.table("sample_commits")

# API request - fetch the table
sample_commits_table = client.get_table(table_ref)

# Preview the first five lines of the table
client.list_rows(sample_commits_table, max_results=5).to_dataframe()

,commit,tree,parent,author,committer,subject,message,trailer,difference,difference_truncated,repo_name,encoding
0,afdba32e2a9ea729a9f9f280dbf6c718773c7ded,d77cca8a096e5320f3194d4a6ca1b4fef2dc9b99,[d65e55d4999b394e37ffe12543ecd2a17b7c44fc],"{'name': 'Jason Gunthorpe', 'email': 'a99b91d7...","{'name': 'Peter Huewe', 'email': '014f16385c5a...",tpm: Pull everything related to /dev/tpmX into...,tpm: Pull everything related to /dev/tpmX into...,"[{'key': 'Signed-off-by', 'value': 'Jason Gunt...","[{'old_mode': 33188, 'new_mode': 33188, 'old_p...",<NA>,torvalds/linux,NaN
1,eb846d9f147455e4e5e1863bfb5e31974bb69b7c,443efbb146c7824508be817923bab04c2185810e,[3af6b35261182ff185db1f0fd271254147e2663e],"{'name': 'Hannes Reinecke', 'email': 'b0d1e9e4...","{'name': 'Christoph Hellwig', 'email': '923f77...",scsi: rename SERVICE_ACTION_IN to SERVICE_ACTI...,scsi: rename SERVICE_ACTION_IN to SERVICE_ACTI...,"[{'key': 'Signed-off-by', 'value': 'Hannes Rei...","[{'old_mode': 33188, 'new_mode': 33188, 'old_p...",<NA>,torvalds/linux,NaN
2,f8798ccbefc0e4ef7438c080b7ba0410738c8cfa,9133440693c02314f1f6f95629b3594ce24ad0f8,[261e767628bb5971b9032439818237cc8511ea94],"{'name': 'Yong Zhang', 'email': '34add0fe16a1f...","{'name': 'Florian Tobias Schandinat', 'email':...",video: irq: Remove IRQF_DISABLED,video: irq: Remove IRQF_DISABLED\n\nSince comm...,"[{'key': 'Signed-off-by', 'value': 'Yong Zhang...","[{'old_mode': 33188, 'new_mode': 33188, 'old_p...",<NA>,torvalds/linux,NaN
3,b83ae6d421435c6204150300f1c25bfbd39cd62b,99c6b661ab7de05c2fd49aa62624d2d6bf8abc69,[de1414a654e66b81b5348dbc5259ecf2fb61655e],"{'name': 'Christoph Hellwig', 'email': '923f77...","{'name': 'Jens Axboe', 'email': 'cd8c6775e60d6...",fs: remove mapping->backing_dev_info,fs: remove mapping->backing_dev_info\n\nNow th...,"[{'key': 'Signed-off-by', 'value': 'Christoph ...","[{'old_mode': 33188, 'new_mode': 33188, 'old_p...",<NA>,torvalds/linux,NaN
4,aaabee8b7686dfe49f10289cb4b7a817b99e5dd9,7ccc6cf829a93d46daf484164a5466c91eca2efa,"[795e9364215dc98b1dea888ebae22383ecbbb92a, 2f2...","{'name': 'Luciano Coelho', 'email': 'd1ef58086...","{'name': 'Luciano Coelho', 'email': 'd1ef58086...",Merge branch 'wl12xx-next' into for-linville,Merge branch 'wl12xx-next' into for-linville\n...,"[{'key': 'Conflicts', 'value': '', 'email': No...","[{'old_mode': 33188, 'new_mode': 33188, 'old_p...",<NA>,torvalds/linux,NaN


In [39]:
# Print information on all the columns in the table
sample_commits_table.schema

[SchemaField('commit', 'STRING', 'NULLABLE', None, None, (), None, None),
 SchemaField('tree', 'STRING', 'NULLABLE', None, None, (), None, None),
 SchemaField('parent', 'STRING', 'REPEATED', None, None, (), None, None),
 SchemaField('author', 'RECORD', 'NULLABLE', None, None, (SchemaField('name', 'STRING', 'NULLABLE', None, None, (), None, None), SchemaField('email', 'STRING', 'NULLABLE', None, None, (), None, None), SchemaField('time_sec', 'INTEGER', 'NULLABLE', None, None, (), None, None), SchemaField('tz_offset', 'INTEGER', 'NULLABLE', None, None, (), None, None), SchemaField('date', 'TIMESTAMP', 'NULLABLE', None, None, (), None, None)), None, None),
 SchemaField('committer', 'RECORD', 'NULLABLE', None, None, (SchemaField('name', 'STRING', 'NULLABLE', None, None, (), None, None), SchemaField('email', 'STRING', 'NULLABLE', None, None, (), None, None), SchemaField('time_sec', 'INTEGER', 'NULLABLE', None, None, (), None, None), SchemaField('tz_offset', 'INTEGER', 'NULLABLE', None, None

In [40]:
# Write a query to find the answer
max_commits_query = """
    SELECT
        -- committer is a RECORD (nested struct), so we use dot notation to reach its child field
        committer.name AS committer_name,

        -- each row in sample_commits is one commit, so COUNT(commit) counts commits per person
        COUNT(commit)  AS num_commits

    FROM `bigquery-public-data.github_repos.sample_commits`

    -- committer.date is a TIMESTAMP; EXTRACT pulls just the year to filter for 2016 commits
    WHERE EXTRACT(YEAR FROM committer.date) = 2016

    -- collapse all rows for the same committer into one row with their total count
    GROUP BY committer_name

    -- most prolific committers appear first
    ORDER BY num_commits DESC
"""

In [41]:
result = client.query(max_commits_query).result().to_dataframe(create_bqstorage_client=False)
result.head()

,committer_name,num_commits
0,Greg Kroah-Hartman,3545
1,David S. Miller,3120
2,TensorFlower Gardener,2449
3,Linus Torvalds,2424
4,Benjamin Pasero,1127


In [42]:
# Construct a reference to the "languages" table
table_ref = dataset_ref.table("languages")

# API request - fetch the table
languages_table = client.get_table(table_ref)

# Preview the first five lines of the table
client.list_rows(languages_table, max_results=5).to_dataframe()

,repo_name,language
0,lemi136/puntovent,"[{'name': 'C', 'bytes': 80}]"
1,taxigps/nctool,"[{'name': 'C', 'bytes': 4461}]"
2,ahy1/strbuf,"[{'name': 'C', 'bytes': 5573}]"
3,nleiten/mod_rpaf-ng,"[{'name': 'C', 'bytes': 30330}]"
4,kmcallister/alameda,"[{'name': 'C', 'bytes': 17077}]"


In [43]:
# Print information on all the columns in the table
languages_table.schema

[SchemaField('repo_name', 'STRING', 'NULLABLE', None, None, (), None, None),
 SchemaField('language', 'RECORD', 'REPEATED', None, None, (SchemaField('name', 'STRING', 'NULLABLE', None, None, (), None, None), SchemaField('bytes', 'INTEGER', 'NULLABLE', None, None, (), None, None)), None, None)]

In [44]:
# Write a query to find the answer
pop_lang_query = """
    SELECT
        -- UNNEST explodes the language array into individual rows, aliased as lang
        -- so we can access lang.name and lang.bytes with dot notation
        lang.name AS language_name,

        -- COUNT(repo_name) counts how many repos use this language
        COUNT(repo_name) AS num_repos

    FROM `bigquery-public-data.github_repos.languages`,
        -- language is REPEATED RECORD (array of structs), so UNNEST is required
        UNNEST(language) AS lang

    -- group all rows with the same language name into one summary row
    GROUP BY language_name

    -- most widely-used languages appear first
    ORDER BY num_repos DESC
"""

In [45]:
result = client.query(pop_lang_query).result().to_dataframe(create_bqstorage_client=False)
result.head()

,language_name,num_repos
0,JavaScript,1099966
1,CSS,807826
2,HTML,777433
3,Shell,640886
4,Python,550905


In [46]:
# Your code here
all_langs_query = """
    SELECT
        -- access child fields of the unnested language struct with dot notation
        lang.name  AS name,
        lang.bytes AS bytes

    FROM `bigquery-public-data.github_repos.languages`,
        -- UNNEST explodes the language array so each language gets its own row
        UNNEST(language) AS lang

    -- restrict to the one repo with the most languages
    WHERE repo_name = 'polyrabbit/polyglot'

    -- languages that take up the most space in the repo appear first
    ORDER BY bytes DESC
"""

In [ ]:
result = client.query(all_langs_query).result().to_dataframe(create_bqstorage_client=False)
result.head()

,name,bytes
0,Lasso,834726
1,C,819142
2,Mercury,709952
3,Objective-C,495392
4,Game Maker Language,298131


: 